In [2]:
import os
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part4c_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")

# 1. Reconstruct 4a features (hypothetical dataset)
pop25_tot = age_sex.groupby("district_id")["population_2025"].sum()
u15_tot = age_sex[age_sex["age_start"] < 15].groupby("district_id")["population_2025"].sum()
o65_tot = age_sex[age_sex["age_start"] >= 65].groupby("district_id")["population_2025"].sum()

df = districts[["district_id", "district_name", "region", "population_2026",
                "arrivals_2025", "departures_2025", "piped_water_pct"]].copy()
df["pop_2025"] = df["district_id"].map(pop25_tot)
df["share_u15_pct"] = (df["district_id"].map(u15_tot) / df["pop_2025"]) * 100.0
df["share_65p_pct"] = (df["district_id"].map(o65_tot) / df["pop_2025"]) * 100.0
exposure_2025 = 0.5 * (df["pop_2025"] + df["population_2026"])
df["nmr_2025_per_1000"] = ((df["arrivals_2025"] - df["departures_2025"]) / exposure_2025) * 1000.0
df["piped_water_imputed"] = df["piped_water_pct"].fillna(df["piped_water_pct"].median())

feat_cols = ["share_u15_pct", "share_65p_pct", "nmr_2025_per_1000", "piped_water_imputed"]
scaler = StandardScaler()
X_std = scaler.fit_transform(df[feat_cols])

# 2. Primary k = 3 model
k_opt = 3
km_primary = KMeans(n_clusters=k_opt, n_init=50, random_state=42)
df["cluster_id"] = km_primary.fit_predict(X_std)

# 3. Centroids back in original units
centroids_orig = pd.DataFrame(scaler.inverse_transform(km_primary.cluster_centers_), columns=feat_cols)
centroids_orig["cluster_id"] = range(k_opt)
centroids_orig["cluster_size"] = [int((df["cluster_id"] == c).sum()) for c in range(k_opt)]
centroids_orig["member_districts"] = [
    ", ".join(df[df["cluster_id"] == c]["district_id"].tolist()) for c in range(k_opt)
]

label_map = {}
for _, r in centroids_orig.iterrows():
    cid = int(r["cluster_id"])
    if r["share_65p_pct"] > 20:
        label_map[cid] = "Older / Out-Migrating"
    elif r["nmr_2025_per_1000"] > 5:
        label_map[cid] = "High-Inflow / High-Water"
    else:
        label_map[cid] = "Young / Low-Water"

centroids_orig["cluster_profile"] = centroids_orig["cluster_id"].map(label_map)
df["cluster_profile"] = df["cluster_id"].map(label_map)

# 4. Stability checks: (a) different random seed, (b) region-median water imputation, (c) complete-case
lbl_seed2 = KMeans(n_clusters=k_opt, n_init=50, random_state=2026).fit_predict(X_std)
ari_seed = adjusted_rand_score(df["cluster_id"], lbl_seed2)

df_reg_imp = df[["share_u15_pct", "share_65p_pct", "nmr_2025_per_1000"]].copy()
df_reg_imp["piped_water_reg_med"] = df["piped_water_pct"].fillna(
    df.groupby("region")["piped_water_pct"].transform("median")
)
lbl_reg_imp = KMeans(n_clusters=k_opt, n_init=50, random_state=42).fit_predict(
    StandardScaler().fit_transform(df_reg_imp)
)
ari_reg_imp = adjusted_rand_score(df["cluster_id"], lbl_reg_imp)

mask_cc = df["piped_water_pct"].notna()
lbl_cc = KMeans(n_clusters=k_opt, n_init=50, random_state=42).fit_predict(
    StandardScaler().fit_transform(df.loc[mask_cc, ["share_u15_pct", "share_65p_pct", "nmr_2025_per_1000", "piped_water_pct"]])
)
ari_cc = adjusted_rand_score(df.loc[mask_cc, "cluster_id"], lbl_cc)

stability_df = pd.DataFrame([
    {"check": "Different random seed (2026 vs 42)", "n_districts": 24, "ari": ari_seed},
    {"check": "Region-median vs overall-median imputation", "n_districts": 24, "ari": ari_reg_imp},
    {"check": "Complete-case (drop 2 missing water districts)", "n_districts": 22, "ari": ari_cc},
])

# Show tables first, then save
print("\n=== CLUSTER CENTROIDS IN ORIGINAL UNITS (k = 3) ===")
display(centroids_orig[["cluster_id", "cluster_profile", "cluster_size"] + feat_cols + ["member_districts"]])
print("=== STABILITY CHECKS ===")
display(stability_df)

safe_save(lambda p: df.to_csv(p, index=False), "part4c_district_clusters_k3.csv")
safe_save(lambda p: centroids_orig.to_csv(p, index=False), "part4c_cluster_centroids_orig_units.csv")
safe_save(lambda p: stability_df.to_csv(p, index=False), "part4c_stability_checks.csv")

BASE_DIR: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-

=== CLUSTER CENTROIDS IN ORIGINAL UNITS (k = 3) ===


,cluster_id,cluster_profile,cluster_size,share_u15_pct,share_65p_pct,nmr_2025_per_1000,piped_water_imputed,member_districts
0,0,Older / Out-Migrating,8,9.491736,27.894612,-1.536355,53.7875,"D03, D06, D09, D12, D15, D18, D21, D24"
1,1,High-Inflow / High-Water,8,12.902920,12.563868,11.860010,75.9750,"D02, D05, D08, D11, D14, D17, D20, D23"
2,2,Young / Low-Water,8,28.971174,14.586556,-2.757829,38.7500,"D01, D04, D07, D10, D13, D16, D19, D22"


=== STABILITY CHECKS ===


,check,n_districts,ari
0,Different random seed (2026 vs 42),24,1.0
1,Region-median vs overall-median imputation,24,1.0
2,Complete-case (drop 2 missing water districts),22,1.0


Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part4c_district_clusters_k3.csv
Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part4c_cluster_centroids_orig_units.csv
Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part4c_stability_checks.csv
